In [1]:
!pip install fastapi uvicorn nest-asyncio

In [6]:
import importlib, sys
if "app" in sys.modules:
    del sys.modules["app"]

In [7]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()  

def run():
    uvicorn.run("app:app", host="0.0.0.0", port=8000)


thread = threading.Thread(target=run, daemon=True)
thread.start()

print("Server running at http://localhost:8000")
print("API docs at     http://localhost:8000/docs")

Server running at http://localhost:8000
API docs at     http://localhost:8000/docs


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INFO:     Started server process [36632]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 8000): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [3]:
import requests

try:
    r = requests.get("http://localhost:8000/")
    print("✅ API alive:", r.json())
except Exception as e:
    print("❌ Still down:", e)

❌ Still down: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001E749282120>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))


In [3]:
import os
print(os.getcwd)

<built-in function getcwd>


In [7]:
print(os.path.exists("app.py"))

True


In [6]:
with open("app.py") as f:
    print(f.read())


from fastapi import FastAPI
from pydantic import BaseModel
from transformers import BertForSequenceClassification, BertTokenizer
import torch

app = FastAPI(title="BERT Sentiment API")

model     = BertForSequenceClassification.from_pretrained("./bert-sentiment-final")
tokenizer = BertTokenizer.from_pretrained("./bert-sentiment-final")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

class SingleRequest(BaseModel):
    text: str

class BatchRequest(BaseModel):
    texts: list[str]

def run_inference(texts):
    tokens = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model(**tokens)

    probs  = torch.softmax(output.logits, dim=1)
    preds  = torch.argmax(probs, dim=1).tolist()
    confs  = probs.max(dim=1).values.tolist()

    label_map = {0: "Negative", 1: "Positive"}
    return [
    

In [8]:
import requests


response = requests.post(
    "http://localhost:8000/predict",
    json={"text": "This product is absolutely amazing!"}
)
print("Single:", response.json())


response = requests.post(
    "http://localhost:8000/predict/batch",
    json={"texts": [
        "Great product, love it!",
        "Terrible quality, waste of money.",
        "It was okay, nothing special."
    ]}
)
print("\nBatch:")
for result in response.json()["results"]:
    print(f"  {result['label']} ({result['confidence']:.2%}) → {result['text']}")

Single: {'text': 'This product is absolutely amazing!', 'label': 'Positive', 'confidence': 0.9858}

Batch:
  Positive (98.51%) → Great product, love it!
  Negative (98.47%) → Terrible quality, waste of money.
  Negative (95.87%) → It was okay, nothing special.


In [8]:
import subprocess, threading, time

def run_ui():
    subprocess.run([
        "streamlit", "run", "streamlit_app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

threading.Thread(target=run_ui, daemon=True).start()
time.sleep(3)
print("✅ Streamlit at http://localhost:8501")

✅ Streamlit at http://localhost:8501
